# cli

> The stream-protocol frontend: the service on stdin/stdout for token-reading clients

In [ ]:
#| default_exp cli

The `clikernel` command connects a text-stream client to one gateway kernel. Requests and replies use a per-process delimiter. Input is not echoed. A `.` line acknowledges each accepted request before its result arrives.

Running the command without arguments creates a Python kernel and stops it on exit. `--kernel` attaches to an existing kernel and leaves it running. Ctrl-C during execution interrupts the kernel without ending the CLI process.


In [ ]:
#| export
import asyncio, secrets, signal, string, sys, termios, threading, traceback, tty
from fastcore.utils import *
from fastcore.script import call_parse
from clikernel.core import Gateway, default_gateway, resolve, session_defaults


In [ ]:
from fastcore.test import *
from io import StringIO
import httpx2, shutil, tempfile
from rustygate.tools import start_gateway


## The protocol

In [ ]:
#| export
_ALPHANUM = string.ascii_letters + string.digits
_MULTILINE = "--"
_MARKER = "loading complete. session delimiter:"


def _new_delim(): return "--" + ''.join(secrets.choice(_ALPHANUM) for _ in range(5))


def _read_block(stdin, delim):
    lines = []
    for line in stdin:
        if line.rstrip("\n") == delim: return "".join(lines), None
        lines.append(line)
    return "", f"missing block terminator: {delim}"


def fmt_error(tag, text):
    nl = '' if text.endswith('\n') else '\n'
    return f"<{tag}>\n{text}{nl}</{tag}>"


A multiline request starts with `--` and ends with the session delimiter on its own line. `_read_block` preserves the cell's line breaks and stops before the next request.

In [ ]:
source, err = _read_block(StringIO("x = 21\nx*2\n--demo\nnext request\n"), '--demo')
test_eq((source, err), ('x = 21\nx*2\n', None))
PrettyString(source)

x = 21
x*2

An unterminated block is a protocol error, not an incomplete cell to execute.

In [ ]:
source, err = _read_block(StringIO('x = 21\n'), '--demo')
test_eq(source, '')
assert 'missing block terminator' in err
err

'missing block terminator: --demo'

`_response` handles one request. It reads any multiline body, rejects malformed framing, and returns `(body, accepted)`. Accepted requests emit `.` before execution. Rejected requests emit no acknowledgement and do not trigger the exit check.

In [ ]:
#| export
def _response(line, stdin, delim, execute):
    "Return (body, accepted), acknowledging accepted requests before execution."
    line = line.rstrip("\n")
    if line == delim: return fmt_error("protocol-error", "no multiline request is open: start one with a bare `--` line"), False
    if line == _MULTILINE:
        code, err = _read_block(stdin, delim)
        if err: return fmt_error("protocol-error", err), False
    elif line.startswith('%%'):
        return fmt_error("protocol-error",
            f"a %% cell magic needs a multiline request; send it as (flush-left):\n    --\n    {line}\n    <rest of cell>\n    {delim}"), False
    else: code = line
    print(".", flush=True)
    try: return execute(code), True
    except BaseException: return fmt_error("internal-error", traceback.format_exc()), True

For a one-line request, the callback supplies the response body. The acknowledgement appears before that result:

In [ ]:
body, accepted = _response('21*2', StringIO(), '--demo', lambda code: str(eval(code)))
test_eq((body, accepted), ('42', True))
body

.


'42'

A cell magic sent without multiline framing is rejected before execution:

In [ ]:
body, accepted = _response('%%time', StringIO(), '--demo', eval)
test_eq(accepted, False)
assert 'a %% cell magic needs a multiline request' in body
PrettyString(body)

<protocol-error>
a %% cell magic needs a multiline request; send it as (flush-left):
    --
    %%time
    <rest of cell>
    --demo
</protocol-error>

In [ ]:
#| export
def _write_response(delim, body=None):
    if body: print(body, end='' if body.endswith('\n') else '\n', flush=True)
    print(delim, flush=True)


def _next_line(stdin):
    "Read one line; when not a TTY, SIGINT while idle means 'interrupt execution', not 'kill the worker', so ignore it"
    while True:
        try: return stdin.readline()
        except KeyboardInterrupt:
            if stdin.isatty(): raise


Terminal input must not echo into protocol output. Terminal setup also disables newline translation and canonical buffering, which can truncate long requests. Signal handling stays enabled. The original terminal settings are restored when the stream ends.

In [ ]:
#| export
def _tty_clear(stream, idx, mask, cc=None):
    "Clear `mask` bits in termios field `idx` when `stream` is a TTY, with optional `cc` char overrides; returns state for `_restore_termios`"
    if not stream.isatty(): return None
    fd = stream.fileno()
    attrs = termios.tcgetattr(fd)
    new_attrs = attrs[:]
    new_attrs[idx] &= ~mask
    if cc:
        new_attrs[6] = attrs[6][:]
        for k, v in cc.items(): new_attrs[6][k] = v
    termios.tcsetattr(fd, termios.TCSADRAIN, new_attrs)
    return fd, attrs


def _restore_termios(state):
    if state: termios.tcsetattr(state[0], termios.TCSADRAIN, state[1])


`serve_stream` announces the startup reply, framing instructions, marker, and delimiter. Each accepted request receives a `.` acknowledgement, its execution result, and the delimiter. Malformed framing returns a `protocol-error` without executing code. A blank line is an empty request and can be used as an idle poll.

In [ ]:
#| export
def serve_stream(
    execute,          # Callable `code -> str`: run one request, returning the rendered response body
    info="",          # Server info announced between the loading lines (forwarded to mcp `instructions`)
    should_exit=None  # Callable checked after each request; truthy stops the worker
):
    "Serve delimiter-framed requests on stdin/stdout."
    # ONLCR off so protocol output stays bare LF; ECHO off (echoed input corrupts the protocol) and ICANON
    # off (canonical mode drops bytes past MAX_CANON with BEL spam; VMIN/VTIME make non-canonical reads
    # return per byte; ISIG stays on so ^C still interrupts)
    output_state = _tty_clear(sys.__stdout__, tty.OFLAG, termios.ONLCR)
    echo_state = _tty_clear(sys.stdin, tty.LFLAG, termios.ECHO | termios.ICANON, {termios.VMIN: 1, termios.VTIME: 0})
    delim = _new_delim()
    print(info, flush=True)
    print(f"<stream-protocol>\nOne-line request: send the line. Each response is an acknowledgement line '.' (request accepted, not complete), the rendered output, then the session delimiter line.\n"
        f"Multiline request (any multi-line cell, including %% cell magics), shown indented -- send it flush-left:\n"
        f"    --\n    <complete cell>\n    {delim}\n"
        f"No IPython prompt, no %cpaste, no invented terminators. A blank line is an empty request, so it doubles as an idle poll. Send 'exit' to end; a fresh process is the restart.\n</stream-protocol>", flush=True)
    print(_MARKER, flush=True)
    _write_response(delim)
    try:
        while True:
            line = _next_line(sys.stdin)
            if not line: break
            body, accepted = _response(line, sys.stdin, delim, execute)
            _write_response(delim, body)
            if accepted and should_exit and should_exit(): break
    finally:
        _restore_termios(echo_state)
        _restore_termios(output_state)

## Running the CLI

`main` runs the asynchronous gateway connection in a background thread. The synchronous protocol loop submits each request with `run_coroutine_threadsafe`. Ctrl-C while waiting sends a kernel interrupt.

The creation or selection reply supplies the kernel id and startup banner announced before the delimiter.


In [ ]:
#| export
_EXITS = ('exit', 'exit()', 'quit', 'quit()')

@call_parse
def main(
    host:str='',    # Gateway: empty for the local default, a `gateways.toml` name, or a URL
    kernel:str='',  # Kernel id (or unique prefix) to attach to; empty creates a kernel, stopped again on exit
):
    "The `clikernel` console script: the stream protocol over one gateway kernel"
    signal.signal(signal.SIGINT, signal.default_int_handler)
    print("please wait, loading...", flush=True)
    loop = asyncio.new_event_loop()
    threading.Thread(target=loop.run_forever, daemon=True).start()
    def run(coro): return asyncio.run_coroutine_threadsafe(coro, loop).result()
    async def _open():
        if not host: return await default_gateway()
        url, token, verify = resolve(host)
        return await Gateway(url, token, verify).initialize(session_defaults(local=False)), None
    g, child = run(_open())
    info = run(g.text('use_kernel', kernel=kernel) if kernel else g.text('create', kernel='ipymini'))
    stop = False
    def execute(code):
        nonlocal stop
        if code.strip() in _EXITS:
            stop = True
            return ''
        fut = asyncio.run_coroutine_threadsafe(g.text('exec', code=code), loop)
        while True:
            try: return fut.result()
            except KeyboardInterrupt: asyncio.run_coroutine_threadsafe(g.call('interrupt'), loop)
    try: serve_stream(execute, info=info, should_exit=lambda: stop)
    finally:
        try: run(g.aclose())
        finally:
            try:
                if child: child.stop()
            finally: loop.call_soon_threadsafe(loop.stop)


The following lessons use the installed `clikernel` command against a disposable gateway. Shared helpers launch the process, read the handshake, and exchange a request. `ask` checks the acknowledgement and returns the response body. It takes the wire text unchanged; multiline framing remains visible in the lesson.


In [ ]:
#| hide
async def read_until(stream, delim):
    lines = []
    while True:
        line = (await asyncio.wait_for(stream.readline(), 30)).decode().rstrip('\n')
        if line == delim: return lines
        lines.append(line)

async def open_cli(*args):
    p = await asyncio.create_subprocess_exec(cmd, *args, env=env,
        stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
    banner = await read_until(p.stdout, _MARKER)
    delim = (await p.stdout.readline()).decode().rstrip('\n')
    return p, delim, banner

async def ask(p, delim, code):
    p.stdin.write((code+'\n').encode())
    await p.stdin.drain()
    test_eq((await p.stdout.readline()).decode().rstrip('\n'), '.')
    return '\n'.join(await read_until(p.stdout, delim))


The examples use an empty configuration directory to avoid running personal startup code.

In [ ]:
tmp = tempfile.TemporaryDirectory()
g = start_gateway()
env = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': tmp.name}
cmd = shutil.which('clikernel')
assert cmd, 'clikernel script not installed'
p, delim, banner = await open_cli()
assert any('created kernel' in line for line in banner)
delim

'--AARhb'

A one-line request needs no framing beyond its newline. The next request sees the same kernel state.

In [ ]:
test_eq(await ask(p, delim, 'x = 21'), '')
answer = await ask(p, delim, 'x*2')
test_eq(answer, '42')
answer


'42'

Cell magics require a multiline request. Send `--`, the complete cell, and the delimiter returned by this process.

In [ ]:
multi = await ask(p, delim, f"--\n%%time\ny = x + 1\n{delim}")
assert 'CPU times' in multi
test_eq(await ask(p, delim, 'y'), '22')
multi

'CPU times: user 1 us, sys: 0 ns, total: 1 us\nWall time: 1.91 us'

Sending `exit` ends the process and stops the kernel it created.

In [ ]:
await ask(p, delim, 'exit')
await p.wait()
remaining = httpx2.get(f'{g.url}/api/kernels').json()
test_eq(remaining, [])
remaining

[]

A kernel created before the CLI survives attachment and exit. `--kernel` accepts a unique id prefix. We use a second process without replacing the first process's variables.

In [ ]:
kid = httpx2.post(f'{g.url}/api/kernels').json()['id']
p2, delim2, _ = await open_cli('--kernel', kid[:8])
test_eq(await ask(p2, delim2, 'z = 99'), '')
await ask(p2, delim2, 'exit')
await p2.wait()
remaining = [k['id'] for k in httpx2.get(f'{g.url}/api/kernels').json()]
test_eq(remaining, [kid])
remaining

['6e73d825a3234ab588086c2c5bb53d96']

In [ ]:
#| hide
httpx2.delete(f'{g.url}/api/kernels/{kid}')
g.stop()
tmp.cleanup()

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()